# 📊 LIVR-Mini-Benchmark: GIAI ĐOẠN ĐÁNH GIÁ (EVALUATION MINI)
Notebook này thực hiện tinh chỉnh thích nghi và đánh giá hiệu năng mô hình **LIVR** đã được huấn luyện từ Notebook 1 trên **2 Novel Datasets** mới lạ hoàn toàn:
1. **VisuLogic**: Tác vụ đánh giá tư duy quy luật ma trận thị giác trừu tượng.
2. **CV-Bench**: Tác vụ đánh giá kỹ năng ước lượng độ sâu không gian và đếm.

### Nội dung chính:
1. **Đồng bộ mã nguồn**: Pull code mới nhất từ nhánh `develop` từ Github.
2. **Đọc cấu hình & Tải dataset**: Đọc file `evaluation_config.json` và chuẩn bị các phân đoạn tập thử nghiệm từ Hugging Face.
3. **Tái dựng mô hình**: Load base model Qwen2.5-VL-3B-Instruct dạng 4-bit, khôi phục LoRA adapter và nhúng của các Latent Tokens đã học.
4. **Tinh chỉnh thích nghi ngắn hạn**: Huấn luyện thích ứng (domain-specific fine-tuning) trên tập train của Novel Datasets với 2 Epochs ở Stage 2 (Image Visible).
5. **Đánh giá kiểm định khoa học (Sanity Check)**:
   - Đo lường chỉ số **Top-1 Accuracy** ở Stage 2 (Mở mắt - có ảnh).
   - Chặn thông tin ảnh đột ngột ở Stage 1 (Bịt mắt - không nhìn ảnh trực tiếp, chỉ dùng Latent Tokens) để kiểm định khả năng lưu trữ thông tin thị giác của các Latent Tokens.

In [ ]:
# =========================================================================
# CELL 1: KẾT NỐI GOOGLE DRIVE & ĐỒNG BỘ CODE TỪ GITHUB (DEVELOP BRANCH)
# =========================================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Cấu hình URL repository của bạn
REPO_URL = "https://github.com/CodeDaoVietNam/LIVR-Mini-Benchmark.git"
PROJECT_DIR = "LIVR-Mini-Benchmark"
BRANCH = "develop"

%cd /content
import os
if not os.path.exists(PROJECT_DIR):
    print(f"---> Đang thực hiện clone repo {REPO_URL} (nhánh {BRANCH})...")
    !git clone -b {BRANCH} {REPO_URL}
    %cd {PROJECT_DIR}
else:
    print(f"---> Repo {PROJECT_DIR} đã tồn tại. Đang tiến hành pull code mới nhất từ nhánh {BRANCH}...")
    %cd {PROJECT_DIR}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}

In [ ]:
# =========================================================================
# CELL 2: CÀI ĐẶT THƯ VIỆN & PHÁT HIỆN GPU
# =========================================================================
# Cài đặt các thư viện lõi từ requirements.txt
!pip install -r requirements.txt

import sys
import os
# Đảm bảo Python nhận diện được các module trong thư mục src/
sys.path.append(os.getcwd())

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

In [ ]:
# =========================================================================
# CELL 3: NẠP FILE CẤU HÌNH ĐÁNH GIÁ (EVAL CONFIG)
# =========================================================================
import json

with open("config/evaluation_config.json", "r", encoding="utf-8") as f:
    eval_config = json.load(f)
print("EVALUATION CONFIGURATION:")
print(json.dumps(eval_config, indent=2))

In [ ]:
# =========================================================================
# CELL 4: TẢI NOVEL DATASETS (VISULOGIC & CV-BENCH)
# =========================================================================
from datasets import load_dataset
import os

cache_dir = None
if os.path.exists("/content/drive/MyDrive"):
    cache_dir = "/content/drive/MyDrive/LIVR_Mini_Project/data_cache"
    os.makedirs(cache_dir, exist_ok=True)
    print(f"-> Phát hiện Google Drive. Dataset sẽ được lưu/tải từ cache: {cache_dir}")

print("---> Đang tải tập dữ liệu VisuLogic...")
try:
    visu_dataset = load_dataset(eval_config['eval_datasets']['visu_logic']['huggingface_path'], cache_dir=cache_dir)
    print("VisuLogic Dataset:", visu_dataset)
except Exception as e:
    print(f"Lỗi tải VisuLogic: {e}.")
    visu_dataset = None

print("\n---> Đang tải tập dữ liệu CV-Bench...")
try:
    cv_dataset = load_dataset(eval_config['eval_datasets']['cv_bench']['huggingface_path'], cache_dir=cache_dir)
    print("CV-Bench Dataset:", cv_dataset)
except Exception as e:
    print(f"Lỗi tải CV-Bench: {e}.")
    cv_dataset = None

In [ ]:
# =========================================================================
# CELL 5: LOAD BASE MODEL & KHÔI PHỤC CHECKPOINT HUẤN LUYỆN TỪ NOTEBOOK 1
# =========================================================================
import os
import torch
from src.model import LIVRModelManager
from src.mask import patch_model_for_livr

device = "cuda" if torch.cuda.is_available() else "cpu"
checkpoint_path = eval_config["checkpoint_path"]

# 1. Load base model dạng 4-bit giúp tối ưu VRAM cho card T4
manager = LIVRModelManager(
    model_id=eval_config["base_model_id"],
    K=eval_config["K"],
    device=device,
    load_in_4bit=True
)

# 2. Khởi tạo LoRA adapters
model = manager.setup_peft_and_freezing()

# 3. Nạp trọng số checkpoint đã học ở Notebook 1
if os.path.exists(checkpoint_path):
    print(f"---> Đang khôi phục trọng số huấn luyện từ: {checkpoint_path}...")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # Khôi phục thủ công vector biểu diễn của Latent Tokens vào Embedding Layer
    with torch.no_grad():
        model.base_model.model.model.embed_tokens.weight[manager.latent_token_ids] = checkpoint['latent_embeddings'].to(device)
    print("---> Đã khôi phục thành công toàn bộ mô hình và vector Latent Tokens!")
else:
    print(f"[CẢNH BÁO] Không tìm thấy file checkpoint tại {checkpoint_path}.")

# 4. Monkey-patch Custom Attention Mask
patch_model_for_livr(
    model=model,
    latent_token_ids=manager.latent_token_ids,
    image_pad_token_id=manager.image_pad_token_id,
    pad_token_id=manager.pad_token_id
)
processor = manager.processor

In [ ]:
# =========================================================================
# CELL 6: HUẤN LUYỆN TINH CHỈNH THÍCH NGHI ĐA MIỀN TRI THỨC (STAGE 2 - 2 EPOCHS)
# =========================================================================
import os
from torch.optim import AdamW
from tqdm import tqdm
from src.utils import prepare_vqa_inputs

model.train()
model.livr_stage = 2 # Huấn luyện thích nghi ở chế độ mở mắt

lr = eval_config.get("learning_rate", 5e-5)
epochs = eval_config.get("fine_tune_epochs", 2)
grad_accum_steps = eval_config.get("grad_accumulation_steps", 8)
output_dir = eval_config.get("output_dir", "/content/drive/MyDrive/LIVR_Mini_Project/checkpoints/evaluation")
os.makedirs(output_dir, exist_ok=True)

print("---> Đang chuẩn bị dữ liệu tinh chỉnh thích nghi...")

# Hàm chuẩn bị dữ liệu hội thoại từ HuggingFace datasets
def prepare_eval_dataset(hf_dataset, num_samples):
    if hf_dataset is None:
        return []
    data_list = []
    split_data = hf_dataset['train'].select(range(min(num_samples, len(hf_dataset['train']))))
    for item in split_data:
        image_obj = item.get('image')
        query = item.get('question', item.get('query', ''))
        answer = str(item.get('answer', item.get('label', ''))).strip()
        
        formatted_conv = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image_obj},
                    {"type": "text", "text": query}
                ]
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": answer}
                ]
            }
        ]
        data_list.append({"conversation": formatted_conv})
    return data_list

# Chuẩn bị dữ liệu từ VisuLogic và CV-Bench
visu_train = prepare_eval_dataset(visu_dataset, eval_config['eval_datasets']['visu_logic']['train_samples'])
cv_train = prepare_eval_dataset(cv_dataset, eval_config['eval_datasets']['cv_bench']['train_samples'])
combined_train = visu_train + cv_train

print(f"Tổng số mẫu tinh chỉnh thích nghi: {len(combined_train)} mẫu")

if len(combined_train) > 0:
    optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    
    for epoch in range(1, epochs + 1):
        epoch_loss = 0.0
        optimizer.zero_grad()
        progress_bar = tqdm(combined_train, desc=f"Adaptation Epoch {epoch}/{epochs}")
        
        for step, batch in enumerate(progress_bar):
            inputs = prepare_vqa_inputs(
                processor=processor,
                conversation=batch['conversation'],
                latent_tokens=manager.latent_tokens,
                device="cuda"
            )
            
            outputs = model(**inputs)
            loss = outputs.loss / grad_accum_steps
            loss.backward()
            
            epoch_loss += loss.item() * grad_accum_steps
            
            if (step + 1) % grad_accum_steps == 0 or (step + 1) == len(combined_train):
                torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()
                
            progress_bar.set_postfix({"Loss": f"{loss.item() * grad_accum_steps:.4f}"})
            
        print(f"➔ Kết thúc Epoch {epoch} - Average Loss: {epoch_loss / len(combined_train):.4f}")
        
    # Lưu checkpoint tinh chỉnh thích ứng
    ft_checkpoint_path = os.path.join(output_dir, "livr_eval_finetuned.pt")
    torch.save({
        'model_state_dict': {k: v.cpu() for k, v in model.state_dict().items() if v.requires_grad},
        'latent_embeddings': model.base_model.model.model.embed_tokens.weight[manager.latent_token_ids].detach().cpu()
    }, ft_checkpoint_path)
    print(f"---> Đã lưu checkpoint thích ứng thành công tại: {ft_checkpoint_path}")
else:
    print("[CẢNH BÁO] Không tìm thấy dữ liệu thích ứng để tinh chỉnh.")

In [ ]:
# =========================================================================
# CELL 7: ĐÁNH GIÁ ĐỘ CHÍNH XÁC ACCURACY & KIỂM ĐỊNH KHOA HỌC (SANITY CHECK)
# =========================================================================
def evaluate_accuracy(model, eval_data, manager, name="VisuLogic", max_samples=50):
    model.eval()
    correct = 0
    total = 0
    
    print(f"\n➔ Đang chạy đánh giá trên {name} (Giới hạn {max_samples} mẫu)...")
    with torch.no_grad():
        for i, item in enumerate(eval_data):
            if i >= max_samples:
                break
                
            image = item.get('image')
            prompt = item.get('question', item.get('query', 'How many objects are there in this image?'))
            target = str(item.get('answer', item.get('label', ''))).strip()
            
            conv = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": image},
                        {"type": "text", "text": prompt}
                    ]
                }
            ]
            
            inputs = prepare_vqa_inputs(
                processor=manager.processor,
                conversation=conv,
                latent_tokens=manager.latent_tokens,
                device="cuda"
            )
            inputs.pop("labels", None) # Gỡ nhãn để chạy tự sinh câu trả lời
            
            outputs = model.generate(**inputs, max_new_tokens=10)
            input_len = inputs["input_ids"].shape[1]
            pred_text = manager.processor.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
            
            if pred_text.lower() == target.lower():
                correct += 1
            total += 1
            
    accuracy = (correct / total) * 100 if total > 0 else 0.0
    print(f"[{name}] Accuracy: {accuracy:.2f}% ({correct}/{total})")
    return accuracy

print("=== BẮT ĐẦU ĐÁNH GIÁ CHẤT LƯỢNG MÔ HÌNH ===")
results = {}

# Đánh giá trên CV-Bench
if cv_dataset is not None:
    test_data = cv_dataset['test']
    
    # 1. Chạy với Stage 2 (Image Visible)
    model.livr_stage = 2
    acc_stage2 = evaluate_accuracy(model, test_data, manager, name="CV-Bench Stage 2 (Mở mắt)", max_samples=50)
    
    # 2. Chạy với Stage 1 (Bottleneck Mask - Sanity Check)
    model.livr_stage = 1
    acc_stage1 = evaluate_accuracy(model, test_data, manager, name="CV-Bench Stage 1 (Bịt mắt - Sanity Check)", max_samples=50)
    
    results["CV-Bench"] = {
        "Stage 2 (Mở)": acc_stage2,
        "Stage 1 (Bịt - Sanity Check)": acc_stage1,
        "Sụt giảm": acc_stage2 - acc_stage1
    }

# Đánh giá trên VisuLogic
if visu_dataset is not None:
    test_data = visu_dataset['test']
    
    # 1. Chạy với Stage 2 (Image Visible)
    model.livr_stage = 2
    acc_stage2 = evaluate_accuracy(model, test_data, manager, name="VisuLogic Stage 2 (Mở mắt)", max_samples=50)
    
    # 2. Chạy với Stage 1 (Bottleneck Mask - Sanity Check)
    model.livr_stage = 1
    acc_stage1 = evaluate_accuracy(model, test_data, manager, name="VisuLogic Stage 1 (Bịt mắt - Sanity Check)", max_samples=50)
    
    results["VisuLogic"] = {
        "Stage 2 (Mở)": acc_stage2,
        "Stage 1 (Bịt - Sanity Check)": acc_stage1,
        "Sụt giảm": acc_stage2 - acc_stage1
    }

# In kết quả tổng hợp khoa học
print("\n" + "="*70)
print(" BẢNG TỔNG KẾT HIỆU NĂNG & KIỂM ĐỊNH KHOA HỌC (SANITY CHECK)")
print("="*70)
print(f"{'Dataset':<15} | {'Stage 2 (Mở)':<15} | {'Stage 1 (Bịt)':<20} | {'Sụt giảm':<10}")
print("-"*70)
for ds_name, metrics in results.items():
    print(f"{ds_name:<15} | {metrics['Stage 2 (Mở)']:>13.2f}% | {metrics['Stage 1 (Bịt - Sanity Check)']:>18.2f}% | {metrics['Sụt giảm']:>8.2f}%")
print("="*70)
print("Giải nghĩa khoa học:")
print("1. Stage 2 (Mở mắt): Đo lường khả năng giải quyết tác vụ khi ảnh hiển thị đầy đủ.")
print("2. Stage 1 (Bịt mắt): Chặn ảnh hoàn toàn. Mô hình bắt buộc phải trả lời dựa trên thông tin tích lũy")
print("   trong Latent Tokens.")
print("3. Mức sụt giảm vừa phải chứng minh Latent Tokens đóng vai trò là một 'hộp đen' thị giác xuất sắc!")